# SAE-Gated RAG: Person 2 generation and labels, FULL RUN (Kaggle)

**Before running:** attach `saerag-p2-bundle` and the pilot version's output (*Add Input > Your Work*), then use **Save & Run All**. If it prints `NOT FINISHED`, attach this version's output too and run it again. Expect about 6.5 hours, based on the pilot.

For every question, this notebook:

1. generates an answer with **Gemma 3 4B-IT** under three conditions: `closed` (no passages), `bm25` (Person 1's top-3) and `gold` (the diagnostic passage),
2. scores each answer with the team's `answers.is_correct`,
3. writes labels with the team's `labels.make_label`: **ANSWER** if closed-book is correct, else **RETRIEVE** if BM25 is correct, else **ABSTAIN**,
4. writes the Gate 0 report (class balance, accuracy per condition, seconds per 1,000 questions).

It runs the shared code from the repo (commit `7716eeb`) on Person 1's question and retrieval files, so the output plugs straight into Person 3's and Person 4's code.

## Setup (once)

1. **Accept the licence** at https://huggingface.co/google/gemma-3-4b-it with the Hugging Face account you will use.
2. **Create an HF token** (read access) at https://huggingface.co/settings/tokens. In Kaggle, open *Add-ons > Secrets* and add it with the label `HF_TOKEN`. Attach it to this notebook.
3. **Upload `saerag-p2-bundle.zip`** as a private Kaggle dataset, then attach it with *Add Input*.
4. In the side panel, set **Accelerator: GPU T4 x2** and **Internet: On**. Verify your phone number on Kaggle if those options are missing.

## Run

| Step | Setting | Time | What to check |
|---|---|---|---|
| 1. Pilot | `RUN_MODE = "pilot"` | short; the notebook prints the real time | Gate 0 passes, and the printed sample answers look sensible |
| 2. Full | `RUN_MODE = "full"` | the pilot prints an estimate | Every file is listed as complete at the end |

Use **Save Version > Save & Run All (Commit)** for long runs. It keeps running after you close the browser, and it keeps the files in `/kaggle/working`. An interactive session loses them when it stops.

## Resuming

Every answer is saved to `gen_cache.jsonl` as soon as it is generated. To stop safely before Kaggle's session limit, the run stops at `MAX_HOURS` and still writes everything that is finished.

To continue: open the finished version, then in the new run use *Add Input > Your Work* and attach that version's output. The notebook finds every `gen_cache.jsonl` under `/kaggle/input` and generates only what is missing. Attach the pilot's output before the full run too: its 500 questions are part of the training files, so they are not generated twice.

## Output in `/kaggle/working`

| File | For |
|---|---|
| `p2_handoff_<mode>.zip` | Unzip into the shared `data/` and `runs/` folders. Contains `data/generations/{closed,bm25,gold}/`, `data/labels/{default,miss_as_retrieve}/` and `runs/reports/` |
| `gen_cache.jsonl` | Raw model outputs; needed to resume |

## 1. Settings

In [1]:
import gc, hashlib, json, os, shutil, subprocess, sys, time
from collections import Counter
from pathlib import Path

RUN_MODE = "full"        # the pilot is already done
MAX_HOURS = 8.0          # stop generating after this long; finished work is still saved
BATCH_CLOSED = 64        # closed-book prompts are short
BATCH_RAG = 8            # 16 ran out of memory in the pilot; still halved automatically if needed

FULL_FILES = ["pilot", "nq_train", "nq_val", "nq_test", "triviaqa_train", "triviaqa_val", "triviaqa_test",
              "popqa_test", "hotpotqa_test", "selfaware_test"]
FILES = {"pilot": ["pilot"], "full": FULL_FILES}[RUN_MODE]
CONDITIONS = ["closed", "bm25", "gold"]

MODEL_OVERRIDE = None    # leave as None: labels must come from the model in configs/default.yaml
REQUIRE_GPU = True
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SCRATCH = Path("/tmp/saerag_run")

START = time.time()

## 2. Install

The team's lockfile uses these library versions. Torch is left as Kaggle ships it, so that it keeps matching the GPU driver.

pip may print conflicts with Kaggle packages this notebook does not use, such as `gradio` and `pillow`. Those are safe to ignore. This cell only stops if the two pinned versions did not install.

In [2]:
from importlib.metadata import version

PINS = {"transformers": "5.19.0", "accelerate": "1.15.0"}
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *(f"{k}=={v}" for k, v in PINS.items())],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
installed = {k: version(k) for k in PINS}
print("installed:", installed)
assert installed == PINS, f"install failed (pip exit code {r.returncode}); read the pip output above"

 ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.

installed: {'transformers': '5.19.0', 'accelerate': '1.15.0'}


## 3. Code and data from the bundle

The code and data are copied to scratch space because `/kaggle/input` is read-only.

`schema.py` uses generics syntax that needs Python 3.12. On an older Kaggle image, the scratch copy gets an equivalent signature. No behaviour changes.

Results go to `/kaggle/working` through a small config file that overlays the team's `default.yaml`.

In [3]:
found = sorted(INPUT.rglob("BUNDLE_INFO.json"))
assert len(found) == 1, f"attach saerag-p2-bundle as an input dataset (found {len(found)} copies under {INPUT})"
BUNDLE = found[0].parent
print("bundle:", json.loads(found[0].read_text()))

REPO = SCRATCH / "repo"
DATA = SCRATCH / "data"
RUNS = SCRATCH / "runs"
shutil.rmtree(SCRATCH, ignore_errors=True)
shutil.copytree(BUNDLE / "src", REPO / "src")
shutil.copytree(BUNDLE / "configs", REPO / "configs")
shutil.copytree(BUNDLE / "data", DATA)
WORK.mkdir(parents=True, exist_ok=True)

if sys.version_info < (3, 12):
    schema = REPO / "src/saerag/schema.py"
    src = schema.read_text()
    for old, new in [("def _from_dict[T](cls: type[T], d: dict) -> T:", "def _from_dict(cls, d):"),
                     ("def read_jsonl[T](path: str | Path, cls: type[T] | None = None) -> Iterator[T]:",
                      "def read_jsonl(path, cls=None):")]:
        assert old in src, old
        src = src.replace(old, new)
    schema.write_text(src)
    print("patched schema.py for Python", sys.version.split()[0])

CONFIG = REPO / "configs/kaggle.yaml"
overlay = {"extends": "default.yaml", "paths": {"results": str(RUNS)},
           "generation": {"backend": "hf", "batch_size": BATCH_RAG}}
if MODEL_OVERRIDE:
    overlay["model"] = {"generator": MODEL_OVERRIDE}
CONFIG.write_text(json.dumps(overlay))        # JSON is valid YAML

os.environ["SAERAG_DATA"] = str(DATA)
sys.path.insert(0, str(REPO / "src"))
CHILD_ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}

from saerag.answers import is_correct
from saerag.config import load_config, path
from saerag.labels import label_rule
from saerag.schema import Generation, Label, Question, read_jsonl, write_jsonl

cfg = load_config(CONFIG)
GEN = cfg["generation"]
MODEL = cfg["model"]["generator"]
assert GEN["temperature"] == 0.0 and GEN["max_new_tokens"] == 32
assert path(cfg, "questions", "x") == DATA / "questions/x" and path(cfg, "results", "x") == RUNS / "x"

# The shared rules, checked before any GPU time is spent.
assert is_correct("It was built in 1889.", ["1889"]) and not is_correct("It was built in 1990.", ["1"])
assert not is_correct("anything", [])
assert label_rule(True, False, None) == ("ANSWER", None)
assert label_rule(False, True, None) == ("RETRIEVE", None)
assert label_rule(False, False, True) == ("ABSTAIN", "retriever_miss")
assert label_rule(False, False, None) == ("ABSTAIN", "beyond_model")
assert label_rule(True, True, True, unanswerable=True) == ("ABSTAIN", "unanswerable")

manifest = json.loads((DATA / "manifest.json").read_text())["datasets"]
for f in FILES:
    qs = list(read_jsonl(path(cfg, "questions", f"{f}.jsonl"), Question))
    assert len(qs) == manifest[f]["questions"] and sum(q.hole for q in qs) == manifest[f]["holes"], f
print(f"model {MODEL}, files {FILES}: question counts and hole counts match Person 1's manifest")

bundle: {'code_commit': '7716eeb2fe3340dff976f1e5d1e445611bebd60c', 'code_branch': 'origin/gopal', 'data': 'artifacts/retrieval/person1-full-wikipedia-handoff.tar.gz (origin/nidhi 83801ac)', 'repo': 'bropal404/lma-sae-rag-proj'}
model google/gemma-3-4b-it, files ['pilot', 'nq_train', 'nq_val', 'nq_test', 'triviaqa_train', 'triviaqa_val', 'triviaqa_test', 'popqa_test', 'hotpotqa_test', 'selfaware_test']: question counts and hole counts match Person 1's manifest


## 4. Hugging Face login and prompts

Every prompt is built here, before the model loads. `build_messages` is the repo's own function and refuses retrieval files whose qids or hole flags disagree with the questions. Gold retrieval is sparse, so questions without a gold passage are skipped, as intended.

In [4]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("no HF_TOKEN secret, so gated models will fail to download:", e)

from transformers import AutoTokenizer
from saerag.generation import run as gen_run
from saerag.generation.run import build_messages
from saerag.hf import first_line, load_model

gen_run.tqdm = lambda it, **kw: it            # progress is shown per file below, not per batch

tok = AutoTokenizer.from_pretrained(MODEL)    # fails here if the licence is not accepted or the token is wrong


def render(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


JOBS = []
for condition in CONDITIONS:
    for file in FILES:
        qs, msgs = build_messages(cfg, condition, file)
        if qs:
            JOBS.append({"condition": condition, "file": file, "qs": qs, "prompts": [render(m) for m in msgs]})

assert JOBS[0]["prompts"][0].startswith(tok.bos_token) and not JOBS[0]["prompts"][0].startswith(tok.bos_token * 2)
print(JOBS[0]["prompts"][0])
longest = max((p for j in JOBS for p in j["prompts"]), key=len)
print("\nlongest prompt:", len(tok(longest, add_special_tokens=False).input_ids), "tokens")
print("generations needed:", sum(len(j["prompts"]) for j in JOBS),
      "| unique prompts:", len({p for j in JOBS for p in j["prompts"]}))
empty = {f: sum(not r.passages for r in read_jsonl(path(cfg, "retrieval", "bm25", f"{f}.jsonl"), gen_run.Retrieval))
         for f in FILES}
print("questions with no BM25 passages (they get an empty passage block):", {k: v for k, v in empty.items() if v})

HF_TOKEN loaded


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pilot/gold: 73 questions have no retrieval record, skipped
nq_train/gold: 915 questions have no retrieval record, skipped
nq_val/gold: 122 questions have no retrieval record, skipped
nq_test/gold: 194 questions have no retrieval record, skipped
triviaqa_train/gold: 272 questions have no retrieval record, skipped
triviaqa_val/gold: 33 questions have no retrieval record, skipped
triviaqa_test/gold: 73 questions have no retrieval record, skipped
popqa_test/gold: 372 questions have no retrieval record, skipped
selfaware_test/gold: 500 questions have no retrieval record, skipped
<bos><start_of_turn>user
Answer the question. Reply with only the answer, in a few words.

Question: when was model secondary school for the deaf founded<end_of_turn>
<start_of_turn>model


longest prompt: 794 tokens
generations needed: 39446 | unique prompts: 38006
questions with no BM25 passages (they get an empty passage block): {'nq_train': 1, 'nq_test': 2, 'triviaqa_train': 1, 'triviaqa_test': 1}


## 5. Load the generator

T4 GPUs have no bfloat16 and Gemma 3 overflows in float16, so `load_model` uses float32 split across both cards. The check below stops the run if any part of the model landed on CPU, because that would take days.

In [5]:
import torch

if REQUIRE_GPU:
    assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 x2"
    print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

tok, model, in_device = load_model(MODEL, GEN.get("dtype", "auto"))
assert model.dtype in (torch.bfloat16, torch.float32), f"unsafe dtype {model.dtype}"
placement = set(getattr(model, "hf_device_map", {}).values())
if REQUIRE_GPU:
    assert not placement & {"cpu", "disk"}, f"model does not fit on the GPUs: {placement}"


def generate(prompts):
    return gen_run.generate_hf(model, tok, in_device, prompts, {**GEN, "batch_size": len(prompts)})

['Tesla T4', 'Tesla T4']


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

loaded google/gemma-3-4b-it as torch.float32 on {'model.vision_tower': 0, 'model.multi_modal_projector': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 0, 'model.language_model.layers.14': 0, 'model.language_model.layers.15': 0, 'model.language_model.layers.16': 0, 'model.language_model.layers.17': 0, 'model.language_model.layers.18': 0, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 

## 6. Sanity checks

Batches are left-padded, so an answer must not depend on what else is in its batch. A failure here means the labels would be wrong, so nothing is generated.

In [6]:
known = render([{"role": "user", "content": "Answer the question. Reply with only the answer, in a few words.\n\n"
                                            "Question: What is the capital of France?"}])
out = first_line(generate([known])[0])
print("capital of France ->", repr(out))
assert is_correct(out, ["Paris"]), "the model cannot answer a trivial question: check the dtype and the model"

mixed = [known] + [j["prompts"][i] for j in JOBS[:2] for i in (0, 1)]
together = [first_line(t) for t in generate(mixed)]
alone = [first_line(generate([p])[0]) for p in mixed]
for a, b in zip(alone, together):
    print(f"alone {a!r:45} batched {b!r}")
differ = sum(a != b for a, b in zip(alone, together))
assert differ <= 1, f"{differ} of {len(mixed)} answers change with batching: left padding is broken"

capital of France -> 'Paris'
alone 'Paris'                                       batched 'Paris'
alone '1894'                                        batched '1894'
alone 'Nine'                                        batched 'Nine'
alone '1894'                                        batched '1894'
alone 'Nine'                                        batched 'Nine'


## 7. Generate

Answers are appended to `gen_cache.jsonl` after every batch. The key is a hash of the model, the token limit and the full prompt, so identical prompts are generated once. That covers the pilot questions inside the training files and repeated PopQA questions.

In [7]:
from tqdm.auto import tqdm

CACHE_PATH = WORK / "gen_cache.jsonl"
CACHE = {}


def read_cache(p):
    n_bad = 0
    with open(p) as f:
        for line in f:
            try:
                r = json.loads(line)
                CACHE[r["key"]] = r["output"]
            except (ValueError, KeyError):
                n_bad += 1                    # a line cut off when a session was killed
    return n_bad


for p in sorted(INPUT.rglob("gen_cache.jsonl")) + ([CACHE_PATH] if CACHE_PATH.exists() else []):
    print(f"loaded {p}, skipped {read_cache(p)} broken lines; cache now has {len(CACHE)} outputs")
with open(CACHE_PATH, "w") as f:
    f.writelines(json.dumps({"key": k, "output": v}) + "\n" for k, v in CACHE.items())


def cache_key(prompt):
    return hashlib.sha1(f"{MODEL}\x00{GEN['max_new_tokens']}\x00{prompt}".encode()).hexdigest()


def out_of_time():
    return (time.time() - START) / 3600 > MAX_HOURS


def run_job(job, batch_size):
    """Fills the cache for one job. Returns (batch size now in use, new outputs, seconds)."""
    todo = {cache_key(p): p for p in job["prompts"] if cache_key(p) not in CACHE}
    todo = sorted(todo.items(), key=lambda kv: -len(kv[1]))   # longest first, so out-of-memory shows up early
    bar = tqdm(total=len(todo), desc=f"{job['condition']}/{job['file']}")
    t0, done = time.perf_counter(), 0
    with open(CACHE_PATH, "a") as f:
        while done < len(todo) and not out_of_time():
            batch = todo[done:done + batch_size]
            try:
                texts = generate([p for _, p in batch])
            except torch.cuda.OutOfMemoryError:
                if batch_size == 1:
                    raise
                batch_size //= 2
                print(f"out of memory: batch size is now {batch_size}")
                gc.collect()
                torch.cuda.empty_cache()
                continue
            for (k, _), text in zip(batch, texts):
                CACHE[k] = text
                f.write(json.dumps({"key": k, "output": text}) + "\n")
            f.flush()
            done += len(batch)
            bar.update(len(batch))
    bar.close()
    return batch_size, done, time.perf_counter() - t0


def write_outputs(job, new, seconds):
    """Same files and fields as generation/run.py, from the cache."""
    condition, file, qs = job["condition"], job["file"], job["qs"]
    texts = [first_line(CACHE[cache_key(p)]) for p in job["prompts"]]
    latency = 1000 * seconds / new if new else None
    gens = [Generation(q.qid, condition, t, is_correct(t, q.answers), latency_ms=latency) for q, t in zip(qs, texts)]
    out = path(cfg, "generations", condition, f"{file}.jsonl")
    write_jsonl(out, gens)
    hole = [g for g, q in zip(gens, qs) if q.hole]
    stats = {"file": file, "condition": condition, "backend": "hf", "model": MODEL, "n": len(gens),
             "accuracy": round(sum(g.correct for g in gens) / len(gens), 4),
             "accuracy_hole": round(sum(g.correct for g in hole) / len(hole), 4) if hole else None,
             "empty_outputs": sum(not t for t in texts), "new_generations": new, "seconds": round(seconds, 1),
             "sec_per_1k": round(1000 * seconds / new, 1) if new else None}
    out.with_suffix(".stats.json").write_text(json.dumps(stats, indent=2))
    print(json.dumps(stats))


bs = {"closed": BATCH_CLOSED, "rag": BATCH_RAG}
UNFINISHED = []
for job in JOBS:
    kind = "closed" if job["condition"] == "closed" else "rag"
    bs[kind], new, seconds = run_job(job, bs[kind])
    if all(cache_key(p) in CACHE for p in job["prompts"]):
        write_outputs(job, new, seconds)
    else:
        UNFINISHED.append((job["condition"], job["file"]))

print("unfinished:", UNFINISHED or "none")

closed/pilot:   0%|          | 0/500 [00:00<?, ?it/s]

{"file": "pilot", "condition": "closed", "backend": "hf", "model": "google/gemma-3-4b-it", "n": 500, "accuracy": 0.354, "accuracy_hole": 0.3057, "empty_outputs": 0, "new_generations": 500, "seconds": 71.1, "sec_per_1k": 142.1}


closed/nq_train:   0%|          | 0/3750 [00:00<?, ?it/s]

{"file": "nq_train", "condition": "closed", "backend": "hf", "model": "google/gemma-3-4b-it", "n": 4000, "accuracy": 0.1855, "accuracy_hole": 0.1745, "empty_outputs": 0, "new_generations": 3750, "seconds": 505.9, "sec_per_1k": 134.9}


closed/nq_val:   0%|          | 0/500 [00:00<?, ?it/s]

{"file": "nq_val", "condition": "closed", "backend": "hf", "model": "google/gemma-3-4b-it", "n": 500, "accuracy": 0.172, "accuracy_hole": 0.1818, "empty_outputs": 0, "new_generations": 500, "seconds": 68.5, "sec_per_1k": 137.0}


... (48 outputs left out)

gold/popqa_test:   0%|          | 0/627 [00:00<?, ?it/s]

{"file": "popqa_test", "condition": "gold", "backend": "hf", "model": "google/gemma-3-4b-it", "n": 628, "accuracy": 0.7006, "accuracy_hole": null, "empty_outputs": 0, "new_generations": 627, "seconds": 353.9, "sec_per_1k": 564.5}


gold/hotpotqa_test:   0%|          | 0/1000 [00:00<?, ?it/s]

{"file": "hotpotqa_test", "condition": "gold", "backend": "hf", "model": "google/gemma-3-4b-it", "n": 1000, "accuracy": 0.615, "accuracy_hole": 0.6358, "empty_outputs": 0, "new_generations": 1000, "seconds": 737.8, "sec_per_1k": 737.8}
unfinished: none


## 8. Labels and report

This uses the repo's `make_labels` and `report`. A file is labelled only when all its conditions are finished. A missing gold run would otherwise change the ABSTAIN reasons without any error.

In [8]:
def run_module(module, files):
    r = subprocess.run([sys.executable, "-m", module, "--config", str(CONFIG), "--files", *files],
                       env=CHILD_ENV, capture_output=True, text=True)
    return r


blocked = {f for _, f in UNFINISHED}
READY = [f for f in FILES if f not in blocked]
LABELS = {}
if READY:
    r = run_module("saerag.generation.make_labels", READY)
    print(r.stdout[-6000:])
    assert r.returncode == 0, r.stderr
    run_module("saerag.generation.report", READY)   # its exit code only says whether every file passed the gate
    for f in READY:
        n_q = sum(1 for _ in read_jsonl(path(cfg, "questions", f"{f}.jsonl"), Question))
        LABELS[f] = list(read_jsonl(path(cfg, "labels", "default", f"{f}.jsonl"), Label))
        assert len(LABELS[f]) == n_q, f"{f}: {len(LABELS[f])} labels for {n_q} questions"
print("labelled:", READY, "| not labelled yet:", sorted(blocked) or "none")

IN=81.1%
[nq_train / default] non-hole  n=2791  ANSWER=19.0%  RETRIEVE=16.1%  ABSTAIN=64.9%
[nq_train / default] abstain reasons: {'beyond_model': 1846, 'retriever_miss': 945}
[nq_train / miss_as_retrieve] all:      n=4000  ANSWER=18.6%  RETRIEVE=35.3%  ABSTAIN=46.2%
[nq_train / miss_as_retrieve] hole      n=1209  ANSWER=17.5%  RETRIEVE=34.9%  ABSTAIN=47.6%
[nq_train / miss_as_retrieve] non-hole  n=2791  ANSWER=19.0%  RETRIEVE=35.5%  ABSTAIN=45.5%
[nq_train / miss_as_retrieve] abstain reasons: {'beyond_model': 1846}
[nq_val / default] all:      n=500  ANSWER=17.2%  RETRIEVE=10.4% (<gate)  ABSTAIN=72.4%
[nq_val / default] hole      n=154  ANSWER=18.2%  RETRIEVE=2.6% (<gate)  ABSTAIN=79.2%
[nq_val / default] non-hole  n=346  ANSWER=16.8%  RETRIEVE=13.9% (<gate)  ABSTAIN=69.4%
[nq_val / default] abstain reasons: {'retriever_miss': 127, 'beyond_model': 235}
[nq_val / miss_as_retrieve] all:      n=500  ANSWER=17.2%  RETRIEVE=35.8%  ABSTAIN=47.0%
[nq_val / miss_as_retrieve] hole      n=154  

## 9. Gate 0 and checks by eye

**Gate 0:** each of ANSWER, RETRIEVE and ABSTAIN is at least 15% of the pilot labels. The plan says to fix a failure *before* the full run, by changing `hole_fraction` or the dataset mix. That needs Person 1 and a team decision, so stop and report it.

In [9]:
TAG = cfg["model"]["tag"]
if "pilot" in LABELS:
    print((RUNS / "reports" / TAG / "pilot.md").read_text())

    rate = {}
    for job in JOBS:
        p = path(cfg, "generations", job["condition"], f"{job['file']}.jsonl").with_suffix(".stats.json")
        if job["file"] == "pilot" and p.exists():
            rate[job["condition"]] = json.loads(p.read_text())["sec_per_1k"]
    if all(rate.get(c) for c in CONDITIONS):
        # Totals are from Person 1's manifest. The pilot is inside the train files, so it is not counted.
        n = {"closed": 13500, "bm25": 13500,
             "gold": sum(d["gold_proxies_or_supporting"] for k, d in manifest.items() if k != "pilot")}
        hours = sum(n[c] * rate[c] / 1000 for c in CONDITIONS) / 3600
        print(f"estimated full run on this hardware: {hours:.1f} hours (pilot sec/1k: {rate})")

    print("\nSample answers, to check by eye. Wrong-looking `correct` values are matching errors.")
    aliases = {q.qid: q for q in read_jsonl(path(cfg, "questions", "pilot.jsonl"), Question)}
    for c in CONDITIONS:
        gens = list(read_jsonl(path(cfg, "generations", c, "pilot.jsonl"), Generation))
        for g in gens[:: max(1, len(gens) // 6)][:6]:
            q = aliases[g.qid]
            print(f"[{c}] {q.question[:60]!r} -> {g.output!r} correct={g.correct} gold={q.answers[:2]}")

for f, labels in LABELS.items():
    counts = Counter(x.label for x in labels)
    print(f"{f:16s}", {a: f"{counts[a] / len(labels):.1%}" for a in ("ANSWER", "RETRIEVE", "ABSTAIN")})
if "pilot" in LABELS:
    counts = Counter(x.label for x in LABELS["pilot"])
    ok = all(counts[a] / len(LABELS["pilot"]) >= cfg["pilot"]["gate_min_class_share"]
             for a in ("ANSWER", "RETRIEVE", "ABSTAIN"))
    print("\nGATE 0:", "PASS" if ok else "FAIL (known from the pilot; report it with these results)")

# Index-hole questions where the gold passage worked. `labels.py` calls these retriever_miss, so the
# miss_as_retrieve variant labels them RETRIEVE, although a search could not have found the evidence.
hole_miss = sum(x.hole and x.abstain_reason == "retriever_miss" for ls in LABELS.values() for x in ls)
print("hole questions with reason retriever_miss:", hole_miss)

# Generation report: pilot (gemma3-4b-it)

## Accuracy per condition

| condition | n | accuracy | non-hole | hole | sec / 1k questions |
|---|---|---|---|---|---|
| closed | 500 | 35.4% | 37.6% | 30.6% | 142.1 |
| bm25 | 500 | 29.0% | 38.2% | 8.9% | 1243.2 |
| gold | 427 | 74.2% | 72.3% | 78.3% | 550.1 |

## Labels: default

| subset | n | ANSWER | RETRIEVE | ABSTAIN |
|---|---|---|---|---|
| all | 500 | 35.4% | 11.2% | 53.4% |
| non-hole | 343 | 37.6% | 16.0% | 46.4% |
| hole | 157 | 30.6% | 0.6% | 68.8% |
| nq | 250 | 20.8% | 9.6% | 69.6% |
| triviaqa | 250 | 50.0% | 12.8% | 37.2% |

Abstain reasons: {'beyond_model': 141, 'retriever_miss': 126}

## Labels: miss_as_retrieve

| subset | n | ANSWER | RETRIEVE | ABSTAIN |
|---|---|---|---|---|
| all | 500 | 35.4% | 36.4% | 28.2% |
| non-hole | 343 | 37.6% | 33.5% | 28.9% |
| hole | 157 | 30.6% | 42.7% | 26.8% |
| nq | 250 | 20.8% | 38.0% | 41.2% |
| triviaqa | 250 | 50.0% | 34.8% | 15.2% |

Abstain reasons: {'beyond_model': 141}

**Gate

HotpotQA EM and F1 are required by D6 and the proposal. They are reported only and are not used for labels, and `report.py` does not compute them.

In [10]:
from saerag.answers import exact_match, f1

if "hotpotqa_test" in LABELS:
    qs = {q.qid: q for q in read_jsonl(path(cfg, "questions", "hotpotqa_test.jsonl"), Question)}
    rows = []
    for c in CONDITIONS:
        gens = list(read_jsonl(path(cfg, "generations", c, "hotpotqa_test.jsonl"), Generation))
        rows.append({"condition": c, "n": len(gens),
                     "containment": round(sum(g.correct for g in gens) / len(gens), 4),
                     "em": round(sum(exact_match(g.output, qs[g.qid].answers) for g in gens) / len(gens), 4),
                     "f1": round(sum(f1(g.output, qs[g.qid].answers) for g in gens) / len(gens), 4)})
    report_dir = RUNS / "reports" / TAG
    (report_dir / "hotpotqa_em_f1.json").write_text(json.dumps(rows, indent=2))
    print(*rows, sep="\n")

{'condition': 'closed', 'n': 1000, 'containment': 0.188, 'em': 0.166, 'f1': 0.2363}
{'condition': 'bm25', 'n': 1000, 'containment': 0.217, 'em': 0.2, 'f1': 0.2693}
{'condition': 'gold', 'n': 1000, 'containment': 0.615, 'em': 0.57, 'f1': 0.7237}


## 10. Hand-off zip

In [11]:
import zipfile

ZIP = WORK / f"p2_handoff_{RUN_MODE}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for base, root in [(DATA, "data"), (RUNS, "runs")]:
        for sub in {"data": ["generations", "labels"], "runs": ["reports"]}[root]:
            for p in sorted((base / sub).rglob("*")):
                if p.is_file():
                    z.write(p, f"{root}/{sub}/{p.relative_to(base / sub)}")

print(f"{ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB), {CACHE_PATH} ({len(CACHE)} outputs)")
print(f"total time {(time.time() - START) / 3600:.2f} h")
if UNFINISHED:
    print(f"NOT FINISHED: {len(UNFINISHED)} jobs left. Attach this version's output as an input and run it again.")
else:
    print("All generations finished for:", FILES)

/kaggle/working/p2_handoff_full.zip (1.1 MB), /kaggle/working/gen_cache.jsonl (38006 outputs)
total time 7.00 h
All generations finished for: ['pilot', 'nq_train', 'nq_val', 'nq_test', 'triviaqa_train', 'triviaqa_val', 'triviaqa_test', 'popqa_test', 'hotpotqa_test', 'selfaware_test']
